# Section 1 — LiveKit Agents (Gemini)
Voice agent with real tool calling via Google Gemini.

**Setup:** Add `GOOGLE_API_KEY` to Colab Secrets (key icon on left sidebar).

In [ ]:
!pip install -q google-genai

In [ ]:
import os
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
print('Key loaded.')

In [ ]:
from google import genai
from google.genai import types
from dataclasses import dataclass

@dataclass
class Transcript:
    text: str

ORDERS = {
    'ORD-001': {'status': 'delivered', 'eta': None, 'restaurant': 'Pizza Palace', 'items': ['Margherita']},
    'ORD-002': {'status': 'in_transit', 'eta': '15 min', 'restaurant': 'Burger Barn', 'items': ['Cheeseburger', 'Fries']},
    'ORD-003': {'status': 'preparing', 'eta': '30 min', 'restaurant': 'Sushi Spot', 'items': ['California Roll']},
}

def get_order_status(order_id: str) -> str:
    o = ORDERS.get(order_id.strip().upper())
    if not o: return f'No order: {order_id}'
    return f"{order_id}: {o['status']}, ETA: {o.get('eta','N/A')}, from {o['restaurant']}"

def cancel_order(order_id: str, reason: str) -> str:
    o = ORDERS.get(order_id.strip().upper())
    if not o: return f"Can't find {order_id}"
    if o['status'] == 'delivered': return 'Already delivered.'
    ORDERS[order_id.upper()]['status'] = 'cancelled'
    return f'{order_id} cancelled. Reason: {reason}. Refund in 3-5 days.'

TOOL_FNS = {'get_order_status': get_order_status, 'cancel_order': cancel_order}
print('Tools defined.')

In [ ]:
# Gemini tool declarations
tools = [types.Tool(function_declarations=[
    types.FunctionDeclaration(name='get_order_status',
        description='Look up order status',
        parameters=types.Schema(type=types.Type.OBJECT,
            properties={'order_id': types.Schema(type=types.Type.STRING)},
            required=['order_id'])),
    types.FunctionDeclaration(name='cancel_order',
        description='Cancel a food order',
        parameters=types.Schema(type=types.Type.OBJECT,
            properties={'order_id': types.Schema(type=types.Type.STRING),
                        'reason': types.Schema(type=types.Type.STRING)},
            required=['order_id', 'reason'])),
])]

client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
chat = client.chats.create(
    model='gemini-2.0-flash',
    config=types.GenerateContentConfig(
        system_instruction='You are a friendly QuickBite support agent. Use tools to look up orders.',
        tools=tools))
print('Gemini chat ready.')

In [ ]:
async def agent_turn(user_text):
    print(f'\n  [user] "{user_text}"')
    response = chat.send_message(user_text)

    # handle tool calls
    for _ in range(5):
        fc = False
        for part in response.candidates[0].content.parts:
            if part.function_call:
                fc = True
                name = part.function_call.name
                args = dict(part.function_call.args)
                print(f'  [tool] {name}({args})')
                result = TOOL_FNS[name](**args)
                print(f'  [result] {result}')
                response = chat.send_message(
                    types.Content(parts=[types.Part(
                        function_response=types.FunctionResponse(
                            name=name, response={'result': result}))]))
                break
        if not fc:
            break

    print(f'  [agent] "{response.text}"')

print('Agent function ready.')

In [ ]:
print('='*55)
print('  SIMULATED VOICE SESSION')
print('='*55)

turns = [
    'Check order ORD-002 for me',
    'When will it arrive?',
    'Also check ORD-003',
    'Cancel ORD-003, changed my mind',
    'What about ORD-999?',
]

for i, t in enumerate(turns, 1):
    print(f'\n--- turn {i} ---')
    await agent_turn(t)

print('\n' + '='*55)
print('Done. Tool calls demonstrated above.')
print('='*55)

## 1.2 Bonus — Swapping providers
The design decouples LLM from STT/TTS:

In [ ]:
print('''
# In LiveKit Agents SDK:

# V1: Deepgram STT + ElevenLabs TTS
class AgentV1(Agent):
    def __init__(self):
        super().__init__(stt=deepgram.STT(), llm=google.LLM(model="gemini-2.0-flash"), tts=elevenlabs.TTS())

# V2: Google Cloud STT + TTS (swap)
class AgentV2(Agent):
    def __init__(self):
        super().__init__(stt=google.STT(), llm=google.LLM(model="gemini-2.0-flash"), tts=google.TTS())

# V3: OpenAI Whisper STT + OpenAI TTS (swap)
class AgentV3(Agent):
    def __init__(self):
        super().__init__(stt=openai.STT(), llm=google.LLM(model="gemini-2.0-flash"), tts=openai.TTS())

# Tools + system prompt UNCHANGED across all versions.
''')